In [3]:
import torch
from sorl.gat_act import GAT, GATConfig

gat_config = GATConfig(
    vocab_sizes=[50304, 8],  # Level 0: 128 tokens, Level 1: 8 abstract tokens
    n_layer=12,
    n_head=6,
    n_embd=768,
    device="cuda" if torch.cuda.is_available() else "cpu"
)

model = GAT(gat_config)


In [4]:
# heuristic rollout
import os, glob, itertools
from pathlib import Path

# MPS specific data loader functional (single device ver.)
# --------------------------------------------
def _load_data_shard(file: Path):
    header = torch.from_file(str(file), False, 256, dtype=torch.int32) # header is 256 int32
    assert header[0] == 20240520, "magic number mismatch in the data .bin file"
    assert header[1] == 1, "unsupported version"
    num_tokens = int(header[2]) # number of tokens (claimed)
    with file.open("rb", buffering=0) as f:
        tokens = torch.empty(num_tokens, dtype=torch.uint16, pin_memory=False) # MPS requires pin_memory=False
        f.seek(256 * 4)
        nbytes = f.readinto(tokens.numpy()) # avoid bytes->array copy by @YouJiacheng
        assert nbytes == 2 * num_tokens, "number of tokens read does not match header"
    return tokens

def data_generator(filename_pattern: str, sequence_length: int, device: str): 

    filename_pattern = "data/fineweb10B/fineweb_train_*.bin"
    files = [Path(file) for file in sorted(glob.glob(filename_pattern))]
    file_iter = itertools.cycle(files)
    tokens, pos = _load_data_shard(next(file_iter)), 0
    while True: 
        # Concern 1. Doesn't this means end-of-file is never reached?
        if pos + sequence_length + 1 >= len(tokens): # not enough data left -> load a new file
            tokens, pos = _load_data_shard(next(file_iter)), 0

        idx = tokens[pos : pos + sequence_length + 1].unsqueeze(0).to(device=device, dtype=torch.int32, non_blocking=True)
        pos += sequence_length
        yield idx

# ------------------------------------------------

# Question 1. Should we separate inputs / targets? 
#             That's really asking whether we want to 'reflect' on inputs, or inputs + next token
#             from the generation perspective, we ought to reflect on inputs and predict next-tok

# Reflection 1. 
# - based on above thought, we ought to modify the 'forward' method to take 'inputs' & 'targets' separately
#   the ._forward_pass and recursion should be done only on 'inputs'


data = _load_data_shard(Path("data/fineweb10B/fineweb_train_000002.bin"))

train_loader = data_generator(filename_pattern="data/fineweb10B/fineweb_train_*.bin", sequence_length=16, device="cpu")
val_loader = data_generator(filename_pattern="data/fineweb10B/fineweb_val_000000.bin", sequence_length=16, device="cpu")


In [8]:
from sorl.neo_utils import sorl_search, compute_loss, sorl_rollout, recursion
from sorl.gat_act import BOS_TOKEN_ID, recursion, infer_level
import torch 
from sorl.neo_utils import select_best_per_doc

# training loop (pseudo version)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.1)

for step in range(100): 
    tokens = next(train_loader)

    # --- mixture of SoRL selection & deep supervision (avg. loss per iteration) ---
    search_tokens, search_ppt, search_adv = sorl_search(tokens, model, n=3, K=3, max_iterations=1, n_continuous=0, memory_span=1024, temperature=1.0)

    # --- compute loss --- 
    traj_loss, abs_loss = compute_loss(search_tokens, model, search_ppt)
    loss = traj_loss
    # --- optimize --- 
    loss.backward() 
    optimizer.step()

    print(f"step {step} loss: {loss.item()} | traj_loss: {traj_loss.item()} | abs_loss: {abs_loss.item()}")

step 0 loss: 14.519709587097168 | traj_loss: 14.519709587097168 | abs_loss: 0.09085465222597122
step 1 loss: 10.091987609863281 | traj_loss: 10.091987609863281 | abs_loss: 0.1003345251083374
step 2 loss: 14.821081161499023 | traj_loss: 14.821081161499023 | abs_loss: 0.1188732385635376
step 3 loss: 14.922721862792969 | traj_loss: 14.922721862792969 | abs_loss: 0.1454056352376938
step 4 loss: 16.265583038330078 | traj_loss: 16.265583038330078 | abs_loss: 0.17087021470069885
step 5 loss: 14.724782943725586 | traj_loss: 14.724782943725586 | abs_loss: 0.18573224544525146
step 6 loss: 11.953681945800781 | traj_loss: 11.953681945800781 | abs_loss: 0.35147443413734436
step 7 loss: 16.16301727294922 | traj_loss: 16.16301727294922 | abs_loss: 0.3129831850528717
step 8 loss: 13.501066207885742 | traj_loss: 13.501066207885742 | abs_loss: 0.5735141038894653
step 9 loss: 15.181537628173828 | traj_loss: 15.181537628173828 | abs_loss: 1.3226038217544556
step 10 loss: 12.408123016357422 | traj_loss: 12

In [ ]:
# Question 1. 
# - there are two ways of inference-time trick with SoRL
# - (1). use recursion, this has causal, parallel property for inference, but requires extra compute in training (include more recursion count)
# - (2). use search, this needs to be done per-token (search on prefix, compute next-token-loss), but no extra compute in training
# - TRM / HRM adopts (1). I also prefer (1). because the search advantage (greedy sample) will be big enough that (2) does not have any gains
# - This simplifies the inference process, too. 

# - Another thought is to re-use sorl_search in validation loop, the ACT halting gadget is also using 


# validation loop (pseudo version)
tokens = next(val_loader)

# === n > 1 includes search | max_iteration > 1 includes recursion | n_continuous doesn't affect causality ===
with torch.no_grad(): 
    aug_tokens, oracle_ppt, oracle_adv = sorl_search(tokens, model, n=3, K=3, max_iterations=1, n_continuous=0, memory_span=1024, temperature=1.0)

traj_loss, abs_loss = compute_loss(aug_tokens, model, oracle_ppt)



# SoRL Rollout Pipeline

The complete SoRL rollout pipeline consists of:

1. **`sorl_rollout`**: Generate n rollouts (1 greedy + n-1 stochastic)
   - Inserts rhythmic placeholder tokens at stride K
   - Fills placeholders using search with different temperatures
   - Returns all n rollouts

2. **`compute_perplexity_per_document`**: Evaluate quality of each rollout
   - Computes perplexity per document for each rollout
   - Lower perplexity = better prediction quality
   - Returns matrix of shape (n_rollouts, n_documents)

3. **`select_best_rollouts`**: Stitch best predictions together
   - Selects best rollout per document (lowest perplexity)
   - Stitches selected segments into final sequence
   - Returns single optimized sequence

This implements the SoRL algorithm where multiple rollouts are sampled and the best segments are selected based on perplexity.
